In [1]:
# GPU 연결 테스트

import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))
    print("GPU 메모리:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")

PyTorch: 2.14.1+cu130
CUDA: True
GPU 이름: Tesla T4
GPU 메모리: 14.56 GB


In [2]:
# 학습 데이터 다운로드

import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

urllib.request.urlretrieve(url, "input.txt")

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("전체 문자 수:", len(text))
print(text[:500])

전체 문자 수: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [3]:
# Tokenizer 구현


# 1. 데이터에 등장하는 모든 문자 수집
chars = sorted(list(set(text)))

# 2. 문자 → 숫자 사전
stoi = {ch: i for i, ch in enumerate(chars)}

# 3. 숫자 → 문자 사전
itos = {i: ch for i, ch in enumerate(chars)}

# 4. 문자 개수
vocab_size = len(chars)


# 5. 문자 → 숫자 변환
def encode(s):
    return [stoi[c] for c in s]


# 6. 숫자 → 문자 변환
def decode(indices):
    return "".join([itos[i] for i in indices])


# 7. 테스트
print("어휘 크기:", vocab_size)
print("문자 변환:", encode("hello"))
print("숫자 복원:", decode(encode("hello")))

어휘 크기: 65
문자 변환: [46, 43, 50, 50, 53]
숫자 복원: hello


In [4]:
# 학습 데이터 생성


import torch

# 전체 문장을 숫자로 변환
data = torch.tensor(encode(text), dtype=torch.long)

# 학습 데이터와 검증 데이터 분리
split = int(0.9 * len(data))

train_data = data[:split]
val_data = data[split:]

print("전체 데이터:", len(data))
print("학습 데이터:", len(train_data))
print("검증 데이터:", len(val_data))

전체 데이터: 1115394
학습 데이터: 1003854
검증 데이터: 111540


In [5]:
# 학습 데이터 배치 생성


# 학습 데이터 배치 생성

# 1. GPU 설정
device = "cuda" if torch.cuda.is_available() else "cpu"

print("실행 장치:", device)


# 2. 학습 설정
batch_size = 64
block_size = 128


# 3. 데이터 배치 생성 함수
def get_batch(split):

    # 학습 데이터 또는 검증 데이터 선택
    source = train_data if split == "train" else val_data

    # 무작위 시작 위치 선택
    ix = torch.randint(
        len(source) - block_size - 1,
        (batch_size,)
    )

    # 입력 데이터 X
    x = torch.stack([
        source[i:i + block_size] for i in ix
    ])

    # 정답 데이터 Y (한 칸 이동)
    y = torch.stack([
        source[i + 1:i + block_size + 1] for i in ix
    ])

    # GPU로 이동
    return x.to(device), y.to(device)


# 4. 학습 데이터 테스트
x, y = get_batch("train")

print("\n입력 X 크기:", x.shape)
print("정답 Y 크기:", y.shape)

print("\n입력 예시:", decode(x[0].tolist()[:30]))
print("정답 예시:", decode(y[0].tolist()[:30]))

실행 장치: cuda

입력 X 크기: torch.Size([64, 128])
정답 Y 크기: torch.Size([64, 128])

입력 예시: ith
all.

ESCALUS:
Look you br
정답 예시: th
all.

ESCALUS:
Look you bri


In [6]:
# Self-Attention 구현

import torch
import torch.nn as nn
import torch.nn.functional as F

class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()

        # Key, Query, Value 생성
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # 미래의 문자를 보지 못하도록 마스킹
        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )

        self.dropout = nn.Dropout(0.1)

    def forward(self, x):

        B, T, C = x.shape

        # Query와 Key 계산
        k = self.key(x)
        q = self.query(x)

        # 어텐션 점수 계산
        wei = q @ k.transpose(-2, -1)
        wei = wei * (k.shape[-1] ** -0.5)

        # 미래 정보 차단
        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        # 확률로 변환
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        # Value를 가중합
        v = self.value(x)
        out = wei @ v

        return out


print("Self-Attention Head 클래스 생성 완료!")

Self-Attention Head 클래스 생성 완료!


In [7]:
# 모델 기본 설정

n_embd = 128

# Token Embedding 생성 후 GPU로 이동
token_embedding_table = nn.Embedding(
    vocab_size,
    n_embd
).to(device)

# Position Embedding 생성 후 GPU로 이동
position_embedding_table = nn.Embedding(
    block_size,
    n_embd
).to(device)


# 입력 데이터 가져오기
x, y = get_batch("train")

B, T = x.shape

# 문자 번호 → 벡터
tok_emb = token_embedding_table(x)

# 위치 번호 생성
pos = torch.arange(T, device=device)

# 위치 번호 → 벡터
pos_emb = position_embedding_table(pos)

# 문자 정보 + 위치 정보
emb = tok_emb + pos_emb


print("Token Embedding 크기:", tok_emb.shape)
print("Position Embedding 크기:", pos_emb.shape)
print("최종 Embedding 크기:", emb.shape)

print("\nToken Embedding 장치:", tok_emb.device)
print("Position Embedding 장치:", pos_emb.device)

Token Embedding 크기: torch.Size([64, 128, 128])
Position Embedding 크기: torch.Size([128, 128])
최종 Embedding 크기: torch.Size([64, 128, 128])

Token Embedding 장치: cuda:0
Position Embedding 장치: cuda:0


In [8]:
# Multi-Head Attention 구현

class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        # 여러 개의 Attention Head 생성
        self.heads = nn.ModuleList([
            Head(head_size) for _ in range(num_heads)
        ])

        # Head들의 결과 연결
        self.proj = nn.Linear(
            num_heads * head_size,
            n_embd
        )

        self.dropout = nn.Dropout(0.1)

    def forward(self, x):

        # 여러 Head의 결과를 연결
        out = torch.cat(
            [head(x) for head in self.heads],
            dim=-1
        )

        out = self.proj(out)
        out = self.dropout(out)

        return out


# 테스트 설정
num_heads = 4
head_size = n_embd // num_heads

# Multi-Head Attention 생성
mha = MultiHeadAttention(
    num_heads,
    head_size
).to(device)

# 테스트
output = mha(emb)

print("Head 개수:", num_heads)
print("Head 하나의 차원:", head_size)
print("입력 크기:", emb.shape)
print("출력 크기:", output.shape)

Head 개수: 4
Head 하나의 차원: 32
입력 크기: torch.Size([64, 128, 128])
출력 크기: torch.Size([64, 128, 128])


In [9]:
# Feed Forward Network (FFN)

class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()

        self.network = nn.Sequential(

            # 128 → 512차원으로 확장
            nn.Linear(n_embd, 4 * n_embd),

            # 비선형성 추가
            nn.ReLU(),

            # 512 → 128차원으로 축소
            nn.Linear(4 * n_embd, n_embd),

            # 과적합 방지
            nn.Dropout(0.1)
        )

    def forward(self, x):
        return self.network(x)


# FFN 생성
ffn = FeedForward(n_embd).to(device)

# 테스트
output = ffn(emb)

print("입력 크기:", emb.shape)
print("출력 크기:", output.shape)
print("실행 장치:", output.device)

입력 크기: torch.Size([64, 128, 128])
출력 크기: torch.Size([64, 128, 128])
실행 장치: cuda:0


In [10]:
# Transformer Block

class Block(nn.Module):

    def __init__(self, n_embd, num_heads):
        super().__init__()

        # Multi-Head Attention
        self.sa = MultiHeadAttention(
            num_heads,
            n_embd // num_heads
        )

        # Feed Forward Network
        self.ffwd = FeedForward(n_embd)

        # Layer Normalization
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):

        # Attention + Residual Connection
        x = x + self.sa(self.ln1(x))

        # FFN + Residual Connection
        x = x + self.ffwd(self.ln2(x))

        return x


# Transformer Block 생성
block = Block(
    n_embd=128,
    num_heads=4
).to(device)


# 테스트
output = block(emb)

print("입력 크기:", emb.shape)
print("출력 크기:", output.shape)
print("실행 장치:", output.device)

입력 크기: torch.Size([64, 128, 128])
출력 크기: torch.Size([64, 128, 128])
실행 장치: cuda:0


In [11]:
# Mini GPT 전체 모델

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        # 문자 임베딩
        self.token_embedding_table = nn.Embedding(
            vocab_size, n_embd
        )

        # 위치 임베딩
        self.position_embedding_table = nn.Embedding(
            block_size, n_embd
        )

        # Transformer Block 4개
        self.blocks = nn.Sequential(
            *[Block(n_embd, num_heads=4) for _ in range(4)]
        )

        # 최종 정규화
        self.ln_f = nn.LayerNorm(n_embd)

        # 다음 문자 예측 출력층
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):

        B, T = idx.shape

        # 1. Token Embedding
        tok_emb = self.token_embedding_table(idx)

        # 2. Position Embedding
        pos = torch.arange(T, device=idx.device)
        pos_emb = self.position_embedding_table(pos)

        # 3. 두 임베딩 결합
        x = tok_emb + pos_emb

        # 4. Transformer Block
        x = self.blocks(x)

        # 5. 최종 정규화
        x = self.ln_f(x)

        # 6. 다음 문자 예측
        logits = self.lm_head(x)

        # 손실 계산
        loss = None

        if targets is not None:
            B, T, C = logits.shape

            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss


# 모델 생성
model = GPTLanguageModel().to(device)

print("Mini GPT 모델 생성 완료!")
print("실행 장치:", device)

# 전체 파라미터 수
total_params = sum(
    p.numel() for p in model.parameters()
)

print("전체 파라미터 수:", f"{total_params:,}")

Mini GPT 모델 생성 완료!
실행 장치: cuda
전체 파라미터 수: 824,897


In [12]:
# 모델 테스트

x, y = get_batch("train")

logits, loss = model(x, y)

print("입력 크기:", x.shape)
print("예측 결과 크기:", logits.shape)
print("초기 손실값:", loss.item())

입력 크기: torch.Size([64, 128])
예측 결과 크기: torch.Size([8192, 65])
초기 손실값: 4.312426567077637


In [13]:
# 학습 설정

learning_rate = 3e-4
max_iters = 10000
eval_interval = 500
eval_iters = 100

# Optimizer: 모델의 파라미터를 업데이트
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)


# 학습 / 검증 손실 측정 함수
@torch.no_grad()
def estimate_loss():

    model.eval()

    results = {}

    for split in ["train", "val"]:

        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):

            x, y = get_batch(split)

            logits, loss = model(x, y)

            losses[k] = loss.item()

        results[split] = losses.mean().item()

    model.train()

    return results


# 학습 전 성능 확인
initial_loss = estimate_loss()

print("학습 전 Train Loss:", initial_loss["train"])
print("학습 전 Val Loss:", initial_loss["val"])
print("Optimizer:", "AdamW")
print("학습 반복 횟수:", max_iters)

학습 전 Train Loss: 4.309175491333008
학습 전 Val Loss: 4.309293746948242
Optimizer: AdamW
학습 반복 횟수: 10000


In [14]:
# Mini GPT 학습 시작

import time

start_time = time.time()

for iter in range(max_iters):

    # 500회마다 학습 및 검증 손실 확인
    if iter % eval_interval == 0:

        losses = estimate_loss()

        print(
            f"Step {iter:4d} | "
            f"Train Loss: {losses['train']:.4f} | "
            f"Val Loss: {losses['val']:.4f}"
        )

    # 학습 데이터 가져오기
    x, y = get_batch("train")

    # 예측 및 손실 계산
    logits, loss = model(x, y)

    # 이전 기울기 초기화
    optimizer.zero_grad(set_to_none=True)

    # 역전파
    loss.backward()

    # 파라미터 업데이트
    optimizer.step()


# 학습 완료
end_time = time.time()

print("\n학습 완료!")
print(f"학습 시간: {(end_time - start_time):.2f}초")

# 최종 성능 평가
final_loss = estimate_loss()

print("\n최종 Train Loss:", final_loss["train"])
print("최종 Val Loss:", final_loss["val"])

Step    0 | Train Loss: 4.3098 | Val Loss: 4.3097
Step  500 | Train Loss: 2.3327 | Val Loss: 2.3441
Step 1000 | Train Loss: 2.0267 | Val Loss: 2.0761
Step 1500 | Train Loss: 1.8366 | Val Loss: 1.9410
Step 2000 | Train Loss: 1.7140 | Val Loss: 1.8589
Step 2500 | Train Loss: 1.6259 | Val Loss: 1.7877
Step 3000 | Train Loss: 1.5682 | Val Loss: 1.7373
Step 3500 | Train Loss: 1.5258 | Val Loss: 1.7015
Step 4000 | Train Loss: 1.4834 | Val Loss: 1.6698
Step 4500 | Train Loss: 1.4652 | Val Loss: 1.6450
Step 5000 | Train Loss: 1.4428 | Val Loss: 1.6341
Step 5500 | Train Loss: 1.4167 | Val Loss: 1.6085
Step 6000 | Train Loss: 1.3997 | Val Loss: 1.5972
Step 6500 | Train Loss: 1.3847 | Val Loss: 1.5856
Step 7000 | Train Loss: 1.3756 | Val Loss: 1.5842
Step 7500 | Train Loss: 1.3647 | Val Loss: 1.5730
Step 8000 | Train Loss: 1.3491 | Val Loss: 1.5665
Step 8500 | Train Loss: 1.3448 | Val Loss: 1.5582
Step 9000 | Train Loss: 1.3338 | Val Loss: 1.5534
Step 9500 | Train Loss: 1.3241 | Val Loss: 1.5464


In [15]:
# 텍스트 생성 함수

@torch.no_grad()
def generate_text(start_text, max_new_tokens=500, temperature=0.8):

    model.eval()

    # 시작 문자를 숫자로 변환
    idx = torch.tensor(
        [encode(start_text)],
        dtype=torch.long,
        device=device
    )

    # 다음 문자 하나씩 생성
    for _ in range(max_new_tokens):

        # 최대 block_size까지만 입력
        idx_cond = idx[:, -block_size:]

        # 다음 문자 예측
        logits, loss = model(idx_cond)

        # 마지막 위치의 예측값 사용
        logits = logits[:, -1, :]

        # Temperature 적용
        logits = logits / temperature

        # 확률로 변환
        probs = F.softmax(logits, dim=-1)

        # 다음 문자 선택
        idx_next = torch.multinomial(probs, num_samples=1)

        # 기존 문장에 문자 추가
        idx = torch.cat((idx, idx_next), dim=1)

    model.train()

    # 숫자를 문자로 복원
    return decode(idx[0].tolist())


# 텍스트 생성 테스트
result = generate_text(
    start_text="First Citizen:",
    max_new_tokens=500
)

print(result)

First Citizen:
How love you bear left with you.

ANGELO:
O lord! The tempester so, run away.

Second King EDWARD:
My heart lie them in his bloody son,
To be see the macry to some poison!
To her.
Now shall be no wear up to the pierce of the and frich,
Are the lands or witness: worthy prince of the
duke, how for what I want to here.

ISABELLA:
Then, that I saw have plain see him she earth,
That love my stand to her action.
And, rind what be must hard; what now, I cannot his hold
No with leave them and my brothe


In [16]:
total_params = sum(
    p.numel() for p in model.parameters()
)

print(f"전체 파라미터 수: {total_params:,}")
print(f"전체 파라미터 수(B): {total_params / 1e9:.6f}B")

전체 파라미터 수: 824,897
전체 파라미터 수(B): 0.000825B
